# Hubworld: Aidalon — Pairing Algorithm Study

A follow-along walkthrough of the `tournament` simulation package. We:

1. Build a player pool with latent **skill** (ground truth the algorithms never see).
2. Run a full event round-by-round under one pairing algorithm.
3. Inspect the records / standings CSV-style.
4. Compare pairing families over many simulated events.
5. Explore **changing the algorithm from round to round**.

See `../PLANNING.md` for scope and `../docs/decisions.md` for the design trade-offs.

In [ ]:
# Make the tournament package importable from the notebooks/ folder.
import sys
from pathlib import Path

MODULE_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(MODULE_ROOT))

from random import Random
import pandas as pd

from tournament import make_players, run_tournament, get_pairing, summary, REGISTRY
from tournament.standings import compute_records
from tournament.metrics import summary as metric_summary
print('pairing algorithms:', list(REGISTRY))

## 1. A player pool

Each player has a normally distributed latent `skill`. Generators use it to play matches; pairing functions only ever see results.

In [ ]:
rng = Random(1)
players = make_players(32, rng, skill_sd=1.0)
pd.DataFrame([(p.pid, p.name, round(p.skill, 3)) for p in players],
             columns=['pid', 'name', 'skill']).head()

## 2. Run one event round-by-round

`run_tournament` pairs round `n` using only the results of rounds `1..n-1`.

In [ ]:
tour = run_tournament(players, n_rounds=5, pairing=get_pairing('adjacent'), rng=rng)

rows = []
for rnd in tour.rounds:
    for m in rnd.matches:
        rows.append({'round': rnd.number, 'a': m.player_a, 'b': m.player_b,
                     'result': f'{m.agents_a}-{m.agents_b}', 'winner': m.winner,
                     'bye': m.is_bye})
matches_df = pd.DataFrame(rows)
matches_df.head(16)

## 3. Final standings

In [ ]:
recs = compute_records(tour)
skill = {p.pid: p.skill for p in players}
standings = pd.DataFrame([{ 'pid': r.pid, 'record': r.record_str,
                            'match_points': r.match_points, 'agent_diff': r.agent_diff,
                            'true_skill': round(skill[r.pid], 3)} for r in recs.values()])
standings = standings.sort_values(['match_points', 'agent_diff'], ascending=False).reset_index(drop=True)
standings.head(10)

In [ ]:
# How well did final standings recover true skill, and how close were pairings?
metric_summary(tour)

## 4. Compare pairing families

Average the metrics over many independent events (shared seeds keep it fair).

In [ ]:
from statistics import fmean

def evaluate(name_or_spec, trials=100, n_players=32, n_rounds=5, base_seed=0):
    spec = get_pairing(name_or_spec) if isinstance(name_or_spec, str) else name_or_spec
    rows = []
    for t in range(trials):
        r = Random(base_seed * 100003 + t)
        ps = make_players(n_players, r)
        rows.append(summary(run_tournament(ps, n_rounds, spec, r)))
    keys = rows[0].keys()
    return {k: fmean(x[k] for x in rows) for k in keys}

comparison = pd.DataFrame({name: evaluate(name, trials=100) for name in REGISTRY}).T
comparison[['mean_skill_gap', 'standings_skill_correlation', 'rematch_count']]

In [ ]:
ax = comparison[['mean_skill_gap', 'standings_skill_correlation']].plot.bar(
    rot=20, figsize=(8, 4), title='Pairing algorithm comparison')
ax.set_ylabel('metric value')

## 5. Changing the algorithm from round to round

Pass a per-round list. Here: random first round (no info yet), then tighten up with `adjacent`/`fold`.

In [ ]:
schedule = [get_pairing('random'), get_pairing('adjacent'),
            get_pairing('adjacent'), get_pairing('fold'), get_pairing('fold')]
print('schedule  :', evaluate(schedule, trials=100))
print('adjacent  :', evaluate('adjacent', trials=100))
print('fold      :', evaluate('fold', trials=100))

### Notes / open questions

- Which metric should be the headline objective? (See `docs/decisions.md` D4.)
- How does each algorithm fare on **comprehensibility** and **manipulability**?
- Try varying `n_players`, `n_rounds`, and `skill_sd` and watch the metrics move.